In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockA (BWCN版 - 修复版)
#
# 修正点：
# 1. parallel=False 解决 NetCDF ID 报错
# 2. 逻辑保持：计算 60-90N 极盖平均部分柱，并基于 Mar-Apr 筛选极端年
# ============================================================

import os
import glob
import numpy as np
import xarray as xr

# ----------------- 配置区域 -----------------
DATA_DIR = "/home/weiji/restart_exam/longrun_B2000WCN_withchem_data/O3"
FILE_PATTERN = "B2000WCN.sample.cam.h3.*.O3.hybrid.nc"

ROOT_DIR = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3_BWCN"
os.makedirs(ROOT_DIR, exist_ok=True)

PRESSURE_RANGES = [
    (30, 70, "30_70hPa"),
    (1, 100, "1_100hPa"),
]

# ----------------- 核心计算函数 -----------------
def calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa, verbose=False):
    """
    Partial O3 column on CAM/WACCM hybrid levels using interface-overlap dp.
    Dask-friendly version (no apply_ufunc).
    """

    import numpy as np
    import xarray as xr

    # ---- constants (SI) ----
    g = 9.80665
    M_air = 28.964 / 1000.0
    Na = 6.02214e23
    factor = Na / (g * M_air * 2.687e20)

    P0 = ds_sub['P0']
    PS = ds_sub['PS']

    # ---- interface pressure (Pa) ----
    P_interface = ds_sub['hyai'] * P0 + ds_sub['hybi'] * PS

    # adjacent interfaces -> rename ilev -> lev
    p_i   = P_interface.isel(ilev=slice(0, -1)).rename({'ilev': 'lev'})
    p_ip1 = P_interface.isel(ilev=slice(1, None)).rename({'ilev': 'lev'})

    # enforce identical lev coordinate (avoid alignment mismatch)
    if 'lev' in ds_sub.coords:
        p_i   = p_i.assign_coords(lev=ds_sub['lev'])
        p_ip1 = p_ip1.assign_coords(lev=ds_sub['lev'])

    # robust layer top/bottom (Pa) -- dask-safe
    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)

    # target bounds (Pa)
    pT = p_top_hpa * 100.0
    pB = p_bot_hpa * 100.0

    # overlap dp (Pa)
    upper = xr.where(p_layer_top > pT, p_layer_top, pT)
    lower = xr.where(p_layer_bot < pB, p_layer_bot, pB)
    overlap = xr.where(lower > upper, lower - upper, 0.0)

    if 'lev' in ds_sub.coords:
        overlap = overlap.assign_coords(lev=ds_sub['lev'])

    # integrate
    O3_col = (ds_sub['O3'] * overlap * factor).sum(dim='lev')

    if verbose:
        dp_eff_hpa = overlap.sum('lev') / 100.0
        # 这里只做整体均值诊断，避免触发大规模计算
        print(f"[Hybrid-Overlap] {p_top_hpa}-{p_bot_hpa} hPa")
        print("  dp_eff global mean (hPa):", float(dp_eff_hpa.mean().compute()))

    return O3_col



def process_block_a():
    search_path = os.path.join(DATA_DIR, FILE_PATTERN)
    files = sorted(glob.glob(search_path))
    if not files:
        # 递归备选
        search_path = os.path.join(DATA_DIR, "**", FILE_PATTERN)
        files = sorted(glob.glob(search_path, recursive=True))
    
    print(f"[BlockA] Found {len(files)} BWCN files.")
    
    # --- 关键修正：parallel=False 防止 HDF5 锁死报错 ---
    print("[BlockA] Opening MFDataset (parallel=False for stability)...")
    ds = xr.open_mfdataset(
        files, 
        combine='by_coords', 
        parallel=False,  # <--- FIXED
        chunks={'time': 365, 'lat': 96, 'lon': 144}
    )

    print("[BlockA] Slicing data to 60-90N to save memory...")
    ds_polar = ds.sel(lat=slice(60, 90))
    
    for ptop, pbot, tag in PRESSURE_RANGES:
        print(f"\n[BlockA] ==== Processing BWCN range {ptop}-{pbot} hPa ({tag}) ====")
        
        PC_TS_NC   = os.path.join(ROOT_DIR, f"O3_pc_BWCN_60_90N_{tag}.nc")
        LOW10_TXT  = os.path.join(ROOT_DIR, f"O3_BWCN_lowest10_years_{tag}.txt")
        LOW25_TXT  = os.path.join(ROOT_DIR, f"O3_BWCN_lowest25pct_years_{tag}.txt")

        print("  Computing partial column on hybrid levels...")
        O3_pc_3d = calc_parc_o3_hybrid(ds_polar, ptop, pbot)
        
        weights = np.cos(np.deg2rad(ds_polar.lat))
        O3_pc_ts = O3_pc_3d.weighted(weights).mean(dim=["lat", "lon"])
        
        print("  Loading data into memory...")
        var_ts = O3_pc_ts.load()
        var_ts.name = f"O3_pc_60_90N_{tag}"
        
        var_ts.to_netcdf(PC_TS_NC)
        print(f"  Saved time series to: {PC_TS_NC}")
        
        # --- 极端年判定逻辑 ---
        # 既然我们关心的是春季极端值，我们按3-4月的数据选年
        # 这些年(Year Y)将作为后续画图的“核心年”，画图时会去取 (Year Y-1) 的秋天
        print("  Computing extreme years (March-April min)...")
        spring = var_ts.sel(time=var_ts.time.dt.month.isin([3, 4]))
        
        # 找出每个春季的最低值
        spring_min_by_year = spring.groupby("time.year").min("time")
        
        spring_years = spring_min_by_year.year.values.astype(int)
        spring_vals = spring_min_by_year.values
        
        idx_sorted = np.argsort(spring_vals)
        
        n_low10 = min(10, len(spring_years))
        lowest10_years = spring_years[idx_sorted[:n_low10]]
        
        n_low25 = max(int(0.25 * len(spring_years)), 1)
        lowest25_years = spring_years[idx_sorted[:n_low25]]
        
        np.savetxt(LOW10_TXT, lowest10_years, fmt="%04d")
        np.savetxt(LOW25_TXT, lowest25_years, fmt="%04d")
        
        print(f"  Lowest 10 years (Event Years): {lowest10_years}")

    print("\n[BlockA] All done.")

if __name__ == "__main__":
    process_block_a()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB (BWCN版 - Nature Style 配色重构版)
#
# 核心升级：
#   * 专业的 Nature Geoscience 风格配色
#   * 气候态背景阴影加深，确保清晰可见
#   * 使用专业的色盲友好配色方案区分极端年份和合成线
#   * 优化字体和线宽设定
# ============================================================

import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
from matplotlib.lines import Line2D
from matplotlib import rcParams

ROOT_DIR = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3_BWCN"

# 定义常数
N_PREV_DAYS = 92 
TOTAL_DAYS = 365 

PRESSURE_RANGES = [
    (30, 70, "30_70hPa"),
    (1, 100, "1_100hPa"),
]

# ----------------- 核心工具函数：拼接跨年数据 (保持不变) -----------------
def get_shifted_lines(da, years_list, days_needed=365):
    """
    输入:
      da: 完整的日数据 TimeSeries (xarray DataArray)
      years_list: "事件年" 列表 (Year Y)
    返回:
      numpy array of shape (N_years, days_needed)
    """
    collected_lines = []
    valid_years = []

    try:
        ts_series = da.to_series()
    except:
        da = da.sel(time=~da.get_index("time").duplicated())
        ts_series = da.to_series()

    for y in years_list:
        y = int(y)
        # 强制4位年份格式，适配 ISO-8601
        prev_y_str = f"{y-1:04d}"
        curr_y_str = f"{y:04d}"
        
        t_prev_start = f"{prev_y_str}-10-01"
        t_prev_end   = f"{prev_y_str}-12-31"
        t_curr_start = f"{curr_y_str}-01-01"
        t_curr_end   = f"{curr_y_str}-09-30"

        try:
            seg_prev = ts_series[t_prev_start : t_prev_end].values
            seg_curr = ts_series[t_curr_start : t_curr_end].values
            combined = np.concatenate([seg_prev, seg_curr])
            
            if len(combined) >= days_needed:
                valid_line = combined[:days_needed]
                collected_lines.append(valid_line)
                valid_years.append(y)
            else:
                pass
        except KeyError:
            continue
            
    return np.array(collected_lines), np.array(valid_years)

# ----------------- 绘图主程序 (配色重构) -----------------
def plot_block_b_nature_style():
    # ================= Nature Style rcParams 设置 =================
    rcParams.update({
        # 优先使用 Helvetica 或 Arial，如果没有则回退到 DejaVu Sans
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 12,          # 稍微调大基础字号
        "axes.linewidth": 1.2,    # 坐标轴线稍微加粗
        "axes.labelsize": 13,
        "axes.titlesize": 14,
        "xtick.major.width": 1.2, #刻度线加粗
        "ytick.major.width": 1.2,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "axes.spines.top": False, # 极简风格，去掉上右边框
        "axes.spines.right": False,
        "legend.fontsize": 10,
        "legend.frameon": False,  # 图例去框
    })

    # ================= 配色方案定义 =================
    # 1. 气候态 (Climatology) - 重点改进：中性灰，足够深以看清
    c_clim_fill = '#999999' # 中灰色
    c_clim_line = '#555555' # 深灰色
    alpha_clim  = 0.3       # 透明度

    # 2. Low 25% Composite - 专业的深蓝色系
    c_low25_line = '#004488' # 深蓝
    c_low25_fill = '#6699CC' # 中蓝
    alpha_low25  = 0.4

    # 3. Extreme Years Top 3 - 精选高对比专业色 (源自色盲友好色板)
    # 颜色1: 朱红, 颜色2: 蓝绿(Teal), 颜色3: 紫红
    spec_colors = ['#D55E00', '#009E73', '#CC79A7'] 

    for ptop, pbot, tag in PRESSURE_RANGES:
        print(f"\n[BlockB] ==== Plotting range {ptop}-{pbot} hPa ({tag}) ====")
        
        PC_TS_NC   = os.path.join(ROOT_DIR, f"O3_pc_BWCN_60_90N_{tag}.nc")
        LOW10_TXT  = os.path.join(ROOT_DIR, f"O3_BWCN_lowest10_years_{tag}.txt")
        LOW25_TXT  = os.path.join(ROOT_DIR, f"O3_BWCN_lowest25pct_years_{tag}.txt")
        
        if not os.path.exists(PC_TS_NC):
            print("  Missing NC file. Run BlockA first.")
            continue

        var_all = xr.load_dataarray(PC_TS_NC)
        
        all_years = np.unique(var_all.time.dt.year.values.astype(int))
        lowest10_years = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
        lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
        
        # 1. 数据准备
        clim_lines, _ = get_shifted_lines(var_all, all_years[1:], days_needed=TOTAL_DAYS)
        clim_mean = np.nanmean(clim_lines, axis=0)
        clim_std  = np.nanstd(clim_lines, axis=0)

        low10_lines, valid_low10_yrs = get_shifted_lines(var_all, lowest10_years, days_needed=TOTAL_DAYS)

        low25_lines, _ = get_shifted_lines(var_all, lowest25_years, days_needed=TOTAL_DAYS)
        low25_mean = np.nanmean(low25_lines, axis=0)
        low25_std  = np.nanstd(low25_lines, axis=0)

        # X 轴设定
        x_axis = np.arange(TOTAL_DAYS)
        # 简化的月份刻度位置 (近似)
        tick_pos = [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 334]
        tick_labels = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

        # ============================================================
        # B1. 图1: Lowest 10 Years (Spaghetti) vs Climatology
        # ============================================================
        fig1, ax1 = plt.subplots(figsize=(12, 7), constrained_layout=True)
        
        # 绘制 Climatology 背景 (新配色)
        ax1.fill_between(x_axis, clim_mean - clim_std, clim_mean + clim_std,
                         color=c_clim_fill, alpha=alpha_clim, 
                         label="Climatology (All Years) ±1$\sigma$", zorder=0)
        ax1.plot(x_axis, clim_mean, color=c_clim_line, lw=2, ls="--", zorder=1)
        
        # 绘制分隔线
        ax1.axvline(x=92, color="k", linestyle=":", linewidth=1.0, alpha=0.5)

        # 绘制 Lowest 10 lines (Spaghetti)
        # 使用 plasma colormap，范围限制在 0-0.85 以避免过于刺眼的黄色
        colors_spaghetti = cm.plasma(np.linspace(0, 0.85, len(low10_lines)))
        
        for i, line in enumerate(low10_lines):
            yr = valid_low10_yrs[i]
            # zorder设高一点确保在线上层
            ax1.plot(x_axis, line, color=colors_spaghetti[i], alpha=0.8, lw=1.5, 
                     label=f"Year {yr}", zorder=2)

        ax1.set_xlim(0, 365)
        ax1.set_xticks(tick_pos)
        ax1.set_xticklabels(tick_labels)
        ax1.set_title(f"Lowest 10 Ozone Events Evolution ({ptop}-{pbot} hPa)\n(Preconditioning: Oct-Dec | Event: Jan-Sep)", 
                      pad=15) # title稍微抬高
        ax1.set_ylabel("Partial Column (DU)")
        # 图例两列显示
        ax1.legend(loc="upper right", ncol=2, borderaxespad=1)
        
        out1 = os.path.join(ROOT_DIR, f"BWCN_O3_lowest10_lines_{tag}_nature.png")
        plt.savefig(out1, dpi=300) # 300dpi 满足出版要求
        print("  Saved Fig1 (Nature Style):", out1)

        # ============================================================
        # B2. 图2: Extremes (Top 3) vs Low 25% Composite
        # ============================================================
        fig2, ax2 = plt.subplots(figsize=(8, 5), constrained_layout=True)
        
        # 背景：Climatology (新配色)
        ax2.fill_between(x_axis, clim_mean - clim_std, clim_mean + clim_std,
                         color=c_clim_fill, alpha=alpha_clim, 
                         label="Climatology ±1$\sigma$", zorder=0)
        ax2.plot(x_axis, clim_mean, color=c_clim_line, lw=1.5, ls="--", zorder=1)

        # 重点：Low 25% Composite (新专业蓝配色)
        # 注意：fill 不加边框 (linewidth=0)，看起来更干净
        ax2.fill_between(x_axis, low25_mean - low25_std, low25_mean + low25_std,
                         color=c_low25_fill, alpha=alpha_low25, linewidth=0,
                         label="Low 25% Composite ±1$\sigma$", zorder=2)
        ax2.plot(x_axis, low25_mean, color=c_low25_line, lw=2.5, zorder=3, 
                 label="Low 25% Mean")

        # 叠加 Top 3 Extreme Years (新专业对比色)
        n_top = min(3, len(low10_lines))
        for i in range(n_top):
            yr = valid_low10_yrs[i]
            line = low10_lines[i]
            # 线宽稍微加粗到 2.5 强调
            ax2.plot(x_axis, line, color=spec_colors[i], lw=2.5, zorder=4, 
                     label=f"Extr Year {yr}")

        ax2.axvline(x=92, color="k", linestyle=":", linewidth=1.0, alpha=0.5)
        ax2.set_xlim(0, 365)
        ax2.set_xticks(tick_pos)
        ax2.set_xticklabels(tick_labels)
        ax2.set_title(f"Composite Analysis & Extremes ({ptop}-{pbot} hPa)", pad=10)
        ax2.set_ylabel("Partial Column (DU)")
        # 图例优化
        ax2.legend(loc="best", frameon=False, borderaxespad=0.5)

        out2 = os.path.join(ROOT_DIR, f"BWCN_O3_composite_vs_extreme_{tag}_nature.png")
        # 保存为 PDF 矢量图，方便后续编辑和高质量出版
        # plt.savefig(out2.replace(".png", ".pdf")) 
        plt.savefig(out2, dpi=300)
        print("  Saved Fig2 (Nature Style):", out2)
        
        plt.close('all')

    print("\n[BlockB] Plotting finished (Nature Style).")

if __name__ == "__main__":
    plot_block_b_nature_style()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Verify hypothesis:
  PLEV calc_parc_o3-style 30–70 (with sparse plev list)
  may behave closer to Hybrid-overlap 20–70 than Hybrid-overlap 30–70.

We compute on ONE day:
  A) Hybrid overlap 30–70
  B) Hybrid overlap 20–70
  C) PLEV calc_parc-style 30–70 (after hybrid->fixed plev interpolation)

We print:
  - dp_eff (hPa)
  - polar-cap mean DU
  - comparisons

Author: you + ChatGPT
"""

import os
import glob
import numpy as np
import xarray as xr

# =========================
# Config
# =========================
DATA_DIR = "/home/weiji/restart_exam/longrun_B2000WCN_withchem_data/O3"
FILE_PATTERN = "B2000WCN.sample.cam.h3.*.O3.hybrid.nc"

PICK_FILE = "/home/weiji/restart_exam/longrun_B2000WCN_withchem_data/O3/B2000WCN.sample.cam.h3.0001.O3.hybrid.nc"
PICK_DATE = None  # e.g., "0001-01-01"

LAT_SLICE = (60, 90)

# Fixed plev list you specified (hPa)
PLEV_LIST_HPA = np.array([
    0.1, 0.5, 1, 2, 3, 5, 10, 20, 30, 50, 70, 100,
    150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000
], dtype=float)

OUT_DIR = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3_BWCN/_debug_verify_20_70_vs_plev"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# Utils
# =========================
def polar_cap_mean(da, lat_name='lat', lon_name='lon'):
    weights = np.cos(np.deg2rad(da[lat_name]))
    return da.weighted(weights).mean(dim=[lat_name, lon_name])

def select_one_day(ds, pick_date=None):
    if pick_date is None:
        time_sel = ds.time.isel(time=0)
        ds_day = ds.isel(time=0)
        print("[Main] PICK_DATE=None, using first time:", str(time_sel.values))
    else:
        ds_tmp = ds.sel(time=pick_date, method='nearest')
        if 'time' in ds_tmp.dims:
            time_sel = ds_tmp.time.isel(time=0)
            ds_day = ds_tmp.isel(time=0)
        else:
            time_sel = ds_tmp['time']
            ds_day = ds_tmp
        print("[Main] Requested date:", pick_date)
        print("[Main] Nearest actual time:", str(time_sel.values))
    return ds_day, time_sel

# =========================
# Hybrid overlap DP
# =========================
def build_hybrid_overlap_components(ds_sub, p_top_hpa, p_bot_hpa):
    g = 9.80665
    M_air = 28.964 / 1000.0
    Na = 6.02214e23
    factor = Na / (g * M_air * 2.687e20)

    P0 = ds_sub['P0']
    PS = ds_sub['PS']

    P_interface = ds_sub['hyai'] * P0 + ds_sub['hybi'] * PS

    p_i   = P_interface.isel(ilev=slice(0, -1)).rename({'ilev': 'lev'})
    p_ip1 = P_interface.isel(ilev=slice(1, None)).rename({'ilev': 'lev'})

    if 'lev' in ds_sub.coords:
        p_i   = p_i.assign_coords(lev=ds_sub['lev'])
        p_ip1 = p_ip1.assign_coords(lev=ds_sub['lev'])

    p_layer_top = xr.apply_ufunc(np.minimum, p_i, p_ip1)
    p_layer_bot = xr.apply_ufunc(np.maximum, p_i, p_ip1)

    pT = p_top_hpa * 100.0
    pB = p_bot_hpa * 100.0

    upper = xr.apply_ufunc(np.maximum, p_layer_top, pT)
    lower = xr.apply_ufunc(np.minimum, p_layer_bot, pB)
    overlap = xr.where(lower > upper, lower - upper, 0.0)

    if 'lev' in ds_sub.coords:
        overlap = overlap.assign_coords(lev=ds_sub['lev'])

    return overlap, factor

def calc_parc_o3_hybrid_overlap(ds_sub, p_top_hpa, p_bot_hpa):
    overlap, factor = build_hybrid_overlap_components(ds_sub, p_top_hpa, p_bot_hpa)
    O3 = ds_sub['O3']
    slab_du = O3 * overlap * factor
    col_2d = slab_du.sum('lev')

    dp_eff_2d = overlap.sum('lev') / 100.0  # hPa
    dp_eff_cap = float(polar_cap_mean(dp_eff_2d))
    col_cap = float(polar_cap_mean(col_2d))

    return col_2d, col_cap, dp_eff_cap

# =========================
# Hybrid -> fixed PLEV interpolation (log-p)
# =========================
def _interp_1d_logp(o3_prof, p_prof, plev_target):
    ok = np.isfinite(o3_prof) & np.isfinite(p_prof)
    if ok.sum() < 2:
        return np.full_like(plev_target, np.nan, dtype=float)

    o3 = o3_prof[ok]
    p  = p_prof[ok]

    idx = np.argsort(p)
    p = p[idx]
    o3 = o3[idx]

    lp = np.log(p)
    lp_t = np.log(plev_target)

    out = np.interp(lp_t, lp, o3)
    out[(plev_target < p.min()) | (plev_target > p.max())] = np.nan
    return out

def interp_o3_hybrid_to_fixed_plev(ds_day, plev_list_hpa):
    P0 = ds_day['P0']
    PS = ds_day['PS']
    P_mid = ds_day['hyam'] * P0 + ds_day['hybm'] * PS  # Pa

    plev_target_pa = xr.DataArray(
        plev_list_hpa * 100.0,
        dims=['plev'],
        coords={'plev': plev_list_hpa}
    )

    O3 = ds_day['O3']

    O3_plev = xr.apply_ufunc(
        _interp_1d_logp,
        O3,
        P_mid,
        plev_target_pa,
        input_core_dims=[['lev'], ['lev'], ['plev']],
        output_core_dims=[['plev']],
        vectorize=True,
        dask='allowed',
        output_dtypes=[float]
    )

    O3_plev = O3_plev.assign_coords(plev=plev_list_hpa)
    O3_plev.name = "O3_on_fixed_plev"
    return O3_plev

# =========================
# PLEV calc_parc_o3-style (match your function)
# =========================
def calc_parc_o3_style_single_day(O3_plev_day, time_sel, p_top_hpa, p_bot_hpa):
    """
    Reproduce your calc_parc_o3 logic on ONE day:
      - build delta_p using adjacent center differences
      - convert to weights_p
      - multiply by 10/(g*m_air)
      - slice(p_top,p_bot)
      - sum
      - /2.687E16

    Return:
      col_2d (DU)
      col_cap (DU)
      dp_eff_implied (hPa)
      selected_plevs (list)
      dp_used_map (dict)
    """

    var = O3_plev_day.expand_dims(time=[time_sel]).assign_coords(time=[time_sel])

    m_air = 28.964/(6.022E23)
    g = 980.6

    if 'plev' in var.dims:
        plev = var.plev
        level = 'plev'
    elif 'lev' in var.dims:
        plev = var.lev
        level = 'lev'
    else:
        plev = var.level
        level = 'level'

    time = var.time
    delta_p = np.zeros((len(time), len(plev)))

    # same unit/orientation rules
    if plev[0] < plev[len(plev)-1] and plev[len(plev)-1] <= 1000:
        factor = 100
        factor_2 = 1
    if plev[0] > plev[len(plev)-1] and plev[0] <= 1000:
        factor = 100
        factor_2 = 1
    if plev[0] < plev[len(plev)-1] and plev[len(plev)-1] > 1000:
        factor = 1
        factor_2 = 100
    if plev[0] > plev[len(plev)-1] and plev[0] > 1000:
        factor = 1
        factor_2 = 100

    ascending = bool(plev[0] < plev[len(plev)-1])

    if ascending:
        for levelx in range(1, len(plev)):
            delta_p[:, levelx].fill(float(plev[levelx] - plev[levelx-1]))
    else:
        for levelx in range(0, len(plev)-1):
            delta_p[:, levelx].fill(float(plev[levelx] - plev[levelx+1]))

    weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])

    O3w = var * weights_p * 10 / (g * m_air)

    # slice exactly like your function
    if ascending:
        slicer = slice(p_top_hpa * factor_2, p_bot_hpa * factor_2)
    else:
        slicer = slice(p_bot_hpa * factor_2, p_top_hpa * factor_2)

    O3w_sel = O3w.sel({level: slicer})

    col_2d = (O3w_sel.sum(level).isel(time=0)) / 2.687E16
    col_cap = float(polar_cap_mean(col_2d))

    # implied dp_eff for reporting
    pvals = plev.values.astype(float)
    delta_1d = np.zeros_like(pvals)

    if ascending:
        delta_1d[1:] = np.diff(pvals)
    else:
        delta_1d[:-1] = pvals[:-1] - pvals[1:]

    if factor == 100:
        dp_hpa = delta_1d
    else:
        dp_hpa = delta_1d / 100.0

    # selected plevs
    sel_plev_vals = O3w_sel[level].values.astype(float).tolist()
    dp_map = {float(pvals[i]): float(dp_hpa[i]) for i in range(len(pvals))}
    dp_eff = float(sum(dp_map[p] for p in sel_plev_vals))

    return col_2d, col_cap, dp_eff, sel_plev_vals, dp_map

# =========================
# Main
# =========================
def main():
    # file
    if PICK_FILE and os.path.exists(PICK_FILE):
        f = PICK_FILE
    else:
        search_path = os.path.join(DATA_DIR, FILE_PATTERN)
        files = sorted(glob.glob(search_path))
        if not files:
            search_path = os.path.join(DATA_DIR, "**", FILE_PATTERN)
            files = sorted(glob.glob(search_path, recursive=True))
        if not files:
            raise FileNotFoundError("No BWCN O3 hybrid files found.")
        f = files[0]

    print("[Main] Using file:", f)

    ds = xr.open_dataset(f)
    print("\n[Dataset Inventory]")
    print("  Variables:", list(ds.data_vars))
    print("  Dims:", dict(ds.sizes))
    print("  Time length:", ds.sizes.get('time', 'N/A'))

    # one day
    ds_day, time_sel = select_one_day(ds, PICK_DATE)

    # polar slice
    lat0, lat1 = LAT_SLICE
    ds_day_polar = ds_day.sel(lat=slice(lat0, lat1))
    print(f"\n[Main] Polar slice lat={lat0}-{lat1}N")
    print("  O3 shape (lev, lat, lon):", ds_day_polar['O3'].shape)

    # -------------------------
    # A) Hybrid overlap 30–70
    # -------------------------
    print("\n==============================")
    print("[A] HYBRID overlap 30–70")
    print("==============================")
    hyb3070_2d, hyb3070_cap, hyb3070_dp = calc_parc_o3_hybrid_overlap(ds_day_polar, 30, 70)
    print(f"  dp_eff (hPa): {hyb3070_dp:.3f}")
    print(f"  Polar-cap mean (DU): {hyb3070_cap:.3f}")

    # -------------------------
    # B) Hybrid overlap 20–70
    # -------------------------
    print("\n==============================")
    print("[B] HYBRID overlap 20–70")
    print("==============================")
    hyb2070_2d, hyb2070_cap, hyb2070_dp = calc_parc_o3_hybrid_overlap(ds_day_polar, 20, 70)
    print(f"  dp_eff (hPa): {hyb2070_dp:.3f}")
    print(f"  Polar-cap mean (DU): {hyb2070_cap:.3f}")

    # -------------------------
    # C) PLEV calc_parc-style 30–70
    # -------------------------
    print("\n==============================")
    print("[C] PLEV calc_parc-style 30–70")
    print("==============================")
    print("  Interpolating hybrid -> fixed plev list...")
    O3_plev_day = interp_o3_hybrid_to_fixed_plev(ds_day_polar, PLEV_LIST_HPA)

    plev3070_2d, plev3070_cap, plev3070_dp, sel_plevs, dp_map = \
        calc_parc_o3_style_single_day(O3_plev_day, time_sel.values, 30, 70)

    print(f"  Selected plevs (hPa): {sel_plevs}")
    print(f"  dp used per selected level (hPa): {[dp_map[p] for p in sel_plevs]}")
    print(f"  IMPLIED dp_eff (hPa): {plev3070_dp:.3f}")
    print(f"  Polar-cap mean (DU): {plev3070_cap:.3f}")

    # -------------------------
    # Summary comparisons
    # -------------------------
    print("\n==============================")
    print("[Summary]")
    print("==============================")
    print(f"  Hybrid 30–70: dp={hyb3070_dp:.3f} hPa, DU={hyb3070_cap:.3f}")
    print(f"  Hybrid 20–70: dp={hyb2070_dp:.3f} hPa, DU={hyb2070_cap:.3f}")
    print(f"  PLEV   30–70: dp={plev3070_dp:.3f} hPa, DU={plev3070_cap:.3f}")

    def rdiff(a, b):
        return (a - b) / b * 100.0 if b != 0 else np.nan

    print("\n  Comparisons vs PLEV(30–70):")
    print(f"    Hybrid(30–70) - PLEV(30–70): {hyb3070_cap-plev3070_cap:.3f} DU  ({rdiff(hyb3070_cap, plev3070_cap):.2f}%)")
    print(f"    Hybrid(20–70) - PLEV(30–70): {hyb2070_cap-plev3070_cap:.3f} DU  ({rdiff(hyb2070_cap, plev3070_cap):.2f}%)")

    print("\n[Done] Verification run complete.")

if __name__ == "__main__":
    main()
